# Phase 3 : Data Quality Analysis

## Connect to Spark

In [3]:
from pyspark.sql import SparkSession

# Build Spark session with S3 + Delta configs
spark = SparkSession.builder \
    .appName("Phase3_DataQuality") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

Spark version: 3.5.0
Spark master: spark://spark-master:7077


26/06/12 12:20:55 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. Load Bronze Layer Data

We read the Delta table stored in MinIO bucket `ecommerce-bronze`.

In [5]:
# Path to the Bronze Delta table in MinIO
bronze_path = "s3a://ecommerce-bronze/delta/bronze/online_retail"

# Read the table
df = spark.read.format("delta").load(bronze_path)

# Total row count and schema overview
total_rows = df.count()
print(f"Total rows: {total_rows:,}")
print("\nSchema:")
df.printSchema()

# Show first 5 rows to verify data
df.show(5, truncate=False)

Total rows: 541,909

Schema:
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)

+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+--------------------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate   |UnitPrice|CustomerID|Country       |ingestion_time            |
+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+--------------------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/2010 8:26|2.55     |17850     |United Kingdom|2026-06-12 08:48:53.736991|
|536

## 2. Null Values Analysis

Check for missing values, especially in `CustomerID` and `Description`.

In [6]:
print("=== NULL VALUE ANALYSIS ===")

# CustomerID nulls
null_customer = df.filter(df.CustomerID.isNull()).count()
print(f"CustomerID NULL: {null_customer:,} ({null_customer/total_rows*100:.2f}%)")

# Description null or empty
null_desc = df.filter(df.Description.isNull() | (df.Description == "")).count()
print(f"Description NULL/Empty: {null_desc:,} ({null_desc/total_rows*100:.2f}%)")

# Per column null counts (only if > 0)
for col_name in df.columns:
    null_count = df.filter(df[col_name].isNull()).count()
    if null_count > 0:
        print(f"  {col_name}: {null_count:,} nulls")

=== NULL VALUE ANALYSIS ===
CustomerID NULL: 135,080 (24.93%)
Description NULL/Empty: 1,454 (0.27%)
  Description: 1,454 nulls
  CustomerID: 135,080 nulls


## 3. Quantity Analysis

Examine negative and zero quantities.  
Negative quantities often indicate cancellations or returns.

In [7]:
print("=== QUANTITY ANALYSIS ===")

# Negative quantities
neg_qty = df.filter(df.Quantity < 0)
neg_count = neg_qty.count()
print(f"Negative Quantity: {neg_count:,} ({neg_count/total_rows*100:.2f}%)")

# Among negatives, distinguish cancellation invoices (InvoiceNo starts with 'C')
cancel_neg = neg_qty.filter(df.InvoiceNo.rlike("^[Cc]")).count()
return_neg = neg_count - cancel_neg
print(f"  Linked to Cancellation (C-code): {cancel_neg:,}")
print(f"  Possible Returns (non C-code): {return_neg:,}")

# Zero quantities
zero_qty = df.filter(df.Quantity == 0).count()
print(f"Zero Quantity: {zero_qty:,}")

# Statistical summary of Quantity
df.select("Quantity").describe().show()

=== QUANTITY ANALYSIS ===
Negative Quantity: 10,624 (1.96%)
  Linked to Cancellation (C-code): 9,288
  Possible Returns (non C-code): 1,336
Zero Quantity: 0
+-------+------------------+
|summary|          Quantity|
+-------+------------------+
|  count|            541909|
|   mean|  9.55224954743324|
| stddev|218.08115785023486|
|    min|            -80995|
|    max|             80995|
+-------+------------------+



## 4. Unit Price Analysis

Identify zero or negative unit prices, which may represent samples, gifts, or erroneous entries.

In [8]:
print("=== UNIT PRICE ANALYSIS ===")

# Zero and negative prices
zero_price = df.filter(df.UnitPrice == 0).count()
neg_price = df.filter(df.UnitPrice < 0).count()
print(f"UnitPrice = 0: {zero_price:,}")
print(f"UnitPrice < 0: {neg_price:,}")
print(f"UnitPrice > 0: {total_rows - zero_price - neg_price:,}")

# Statistical summary
df.select("UnitPrice").describe().show()

=== UNIT PRICE ANALYSIS ===
UnitPrice = 0: 2,515
UnitPrice < 0: 2
UnitPrice > 0: 539,392
+-------+-----------------+
|summary|        UnitPrice|
+-------+-----------------+
|  count|           541909|
|   mean|4.611113626082972|
| stddev| 96.7598530611797|
|    min|        -11062.06|
|    max|          38970.0|
+-------+-----------------+



## 5. Duplicate Analysis

Detect exact and logical duplicates.  
Logical duplicates exclude the `ingestion_time` column, as it differs between runs.

In [9]:
print("=== DUPLICATE ANALYSIS ===")

# Exact duplicates (all columns identical)
exact_dup = df.count() - df.distinct().count()
print(f"Exact duplicate rows: {exact_dup:,}")

# Duplicates ignoring ingestion_time
cols_no_ingestion = [c for c in df.columns if c != "ingestion_time"]
logical_dup = df.count() - df.select(cols_no_ingestion).distinct().count()
print(f"Logical duplicates (excluding ingestion_time): {logical_dup:,}")

=== DUPLICATE ANALYSIS ===


Exact duplicate rows: 5,268


[Stage 129:============================>                            (1 + 1) / 2]

Logical duplicates (excluding ingestion_time): 5,268


## 6. StockCode Analysis

Check for non-numeric StockCodes (e.g., POST, DOT, M).  
These codes usually represent shipping charges, discounts, or manual adjustments, not actual products.

In [10]:
print("=== STOCKCODE ANALYSIS ===")

unique_stock = df.select("StockCode").distinct().count()
print(f"Unique StockCode values: {unique_stock:,}")

# Non-numeric StockCodes (does not consist of only digits)
non_num = df.filter(~df.StockCode.rlike("^[0-9]+$"))
non_num_count = non_num.count()
print(f"Non-numeric StockCode (e.g., POST, DOT): {non_num_count:,} records")

if non_num_count > 0:
    print("\nExamples of non-numeric StockCodes:")
    non_num.select("StockCode", "Description").distinct().show(10, truncate=False)

=== STOCKCODE ANALYSIS ===
Unique StockCode values: 4,070
Non-numeric StockCode (e.g., POST, DOT): 54,873 records

Examples of non-numeric StockCodes:
+---------+----------------------------------+
|StockCode|Description                       |
+---------+----------------------------------+
|84279P   |CHERRY BLOSSOM  DECORATIVE FLASK  |
|90184A   |AMBER CHUNKY BEAD BRACELET W STRAP|
|47593A   |CAROUSEL PONIES BABY BIB          |
|35637C   |PINK STRING CURTAIN WITH POLE     |
|37444A   |YELLOW BREAKFAST CUP AND SAUCER   |
|47559B   |TEA TIME OVEN GLOVE               |
|90013B   |BLACK VINTAGE EARRINGS            |
|84596e   |SMALL LICORICE DES PINK BOWL      |
|85129B   |BEADED CRYSTAL HEART GREEN SMALL  |
|85035c   |ROSE 3 WICK MORRIS BOX CANDLE     |
+---------+----------------------------------+
only showing top 10 rows



## 7. Cancellation Invoice Analysis

Invoices starting with 'C' are cancellations.  
We count how many records belong to such invoices.

In [11]:
print("=== CANCELLATION INVOICE ANALYSIS ===")

cancel_inv = df.filter(df.InvoiceNo.rlike("^[Cc]"))
cancel_count = cancel_inv.count()
print(f"Cancellation invoice records: {cancel_count:,} ({cancel_count/total_rows*100:.2f}%)")

=== CANCELLATION INVOICE ANALYSIS ===
Cancellation invoice records: 9,288 (1.71%)


## 8. Summary of Findings

- **Missing CustomerID**: ~25% of records. This is substantial; rather than dropping, we should isolate these records.
- **Negative Quantities**: Mostly linked to cancellation invoices. A small fraction may be returns or bad data.
- **Zero UnitPrice**: Very few records; likely samples or gifts – needs flagging.
- **Exact Duplicates**: Some duplicates exist; they should be removed in the Silver layer.
- **Non-numeric StockCodes**: Several administrative codes (POST, DOT, etc.) should be separated from product-level analysis.

These findings will guide the cleaning and transformation rules in **Phase 4: Silver Layer Processing**.